In [5]:
import math
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [6]:

df = pd.read_csv('cleaned_data/information/panel_macro_cleaned.csv') 
latest_year = int(df['Year'].max())
focus_countries = ['SGP', 'IRL', 'USA', 'IND', 'DEU', 'CHN', 'JPN', 'VNM', 'THA', 'PHL', 'ZAF']

bubble_df = df[(df['Year'] == latest_year) & (df['Country'].isin(focus_countries))].copy()
bubble_df['GDP_Billion_USD'] = bubble_df['GDP_Current_USD'] / 1e9

# Điền 0 cho các giá trị NaN của GDP per capita để tránh lỗi khi vẽ size
bubble_df['GDP_Per_Capita_USD'] = bubble_df['GDP_Per_Capita_USD'].fillna(0)

# 2. Vẽ biểu đồ bằng Plotly Express
fig_bubble = px.scatter(
    bubble_df,
    x='GDP_Billion_USD',
    y='Economic_Openness_Pct',
    size='GDP_Per_Capita_USD',
    color='GDP_Per_Capita_USD',
    text='Country',
    hover_name='Country',
    size_max=60,
    color_continuous_scale='Viridis',
    title=f"BIỂU ĐỒ BONG BÓNG: ĐỘ MỞ VS QUY MÔ (NĂM {latest_year})",
    labels={
        'GDP_Billion_USD': 'GDP (Tỷ USD)',
        'Economic_Openness_Pct': 'Độ mở kinh tế (% GDP)',
        'GDP_Per_Capita_USD': 'GDP/Người (USD)'
    }
)

# 3. Tùy chỉnh hiển thị (Label, Tooltip, Layout)
fig_bubble.update_traces(
    textposition='top center',
    marker=dict(line=dict(width=1, color='white')),
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "GDP: %{x:,.1f} Tỷ USD<br>"
        "Độ mở: %{y:,.1f} %<br>"
        "GDP/Người: %{marker.size:,.0f} USD<extra></extra>" 
        # (Lưu ý: trong px, marker.size lưu giá trị thực tế truyền vào)
    ),
)

fig_bubble.update_layout(
    template="plotly_white",
    height=700,
    margin=dict(l=60, r=40, t=80, b=60),
    coloraxis_colorbar=dict(title="GDP/Người (USD)")
)

# Thêm tuỳ chỉnh Log cho trục X nếu bong bóng của Mỹ và Trung Quốc làm biểu đồ bị dồn (Bỏ comment nếu muốn dùng)
fig_bubble.update_xaxes(type="log", title="GDP (Tỷ USD) - Thang đo Logarit")

fig_bubble.show(config={'responsive': True})
fig_bubble.write_html("fig/bubble_chart_openness_gdp.html")

In [7]:
df = pd.read_csv('cleaned_data/VNM_macro_cleaned.csv')

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Thêm FDI Inflows dưới dạng cột
fig.add_trace(
    go.Bar(x=df['Year'], y=df['FDI_Inflows_USD'], name="Vốn FDI (USD)", marker_color="#2FA1FF"),
    secondary_y=False,
)

# Thêm Xuất khẩu dưới dạng đường
fig.add_trace(
    go.Scatter(x=df['Year'], y=df['Exports_USD'], name="Kim ngạch Xuất khẩu (USD)", line=dict(color="#FF4A4A", width=3)),
    secondary_y=True,
)

fig.update_layout(
    title_text="<b>Tương quan giữa dòng vốn FDI và Tăng trưởng Xuất khẩu Việt Nam (1990-2024)</b>",
    hovermode="x unified"
)

fig.update_xaxes(title_text="Năm")
fig.update_yaxes(title_text="FDI Inflows (Current USD)", secondary_y=False)
fig.update_yaxes(title_text="Exports (Current USD)", secondary_y=True)

fig.show()

In [9]:
# Ve toan bo chi so theo thoi gian (Viet Nam)
file_path = "cleaned_data/VNM_macro_cleaned.csv"
df_all = pd.read_csv(file_path)

indicator_map = {
    "FDI_Inflows_USD": "Dòng vốn FDI ròng (USD hiện hành)",
    "Remittances_Pct_GDP": "Tỷ trọng Kiều hối (% GDP)",
    "Inflation_CPI_Pct": "Tỷ lệ Lạm phát (CPI, %)",
    "Lending_Interest_Rate_Pct": "Lãi suất cho vay (%)",
    "Exports_USD": "Kim ngạch Xuất khẩu (USD hiện hành)",
    "Imports_USD": "Kim ngạch Nhập khẩu (USD hiện hành)",
    "GDP_Growth_Pct": "Tốc độ Tăng trưởng GDP (%)",
    "Unemployment_Pct": "Tỷ lệ Thất nghiệp (%)",
    "Population": "Tổng dân số (Người)",
    "GNI_USD": "Tổng thu nhập quốc gia - GNI (USD)",
    "GDP_Current_USD": "Quy mô GDP (USD hiện hành)",
    "Labor_Force_Total": "Lực lượng lao động (Người)",
    "Trade_Balance_USD": "Cán cân Thương mại (USD)",
    "GDP_Per_Capita_USD": "GDP bình quân đầu người (USD)",
    "GDP_GNI_Gap_Pct": "Khoảng cách GDP - GNI (% GDP)",
    "Economic_Openness_Pct": "Độ mở Kinh tế (% GDP)",
    "FDI_to_GDP_Pct": "Tỷ lệ vốn FDI / GDP (%)",
    "Labor_Participation_Rate_Pct": "Tỷ lệ tham gia Lực lượng lao động (%)"
}

indicators = [col for col in indicator_map.keys() if col in df_all.columns]
cols = 2
rows = int(math.ceil(len(indicators) / cols))

fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f"{col}: {indicator_map[col]}" for col in indicators],
    vertical_spacing=0.05,
    horizontal_spacing=0.08,
)

for i, col_name in enumerate(indicators):
    row = (i // cols) + 1
    col = (i % cols) + 1

    fig.add_trace(
        go.Scatter(
            x=df_all["Year"],
            y=df_all[col_name],
            mode="lines+markers",
            name=col_name,
            line=dict(width=2),
            hovertemplate="Nam: %{x}<br>Gia tri: %{y:,.2f}<extra></extra>",
            showlegend=False,
        ),
        row=row, col=col,
    )

    fig.update_xaxes(title_text="Năm", row=row, col=col)
    fig.update_yaxes(title_text=col_name, row=row, col=col)

fig.update_layout(
    height=300 * rows,
    title_text="Các chỉ số kinh tế vĩ mô của Việt Nam theo thời gian",
    template="plotly_white",
    margin=dict(l=60, r=40, t=80, b=40),
)

fig.show(config={"responsive": True})
fig.write_html("fig/VNM_Macro_Stats.html")